# First Look at the Soccer Data

Run `python scripts/load_data.py` first to populate the DB.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src.db import get_connection
conn = get_connection()
for table in ['competitions', 'teams', 'matches']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {table}', conn).iloc[0]['n']
    print(f'{table:15s} {n:>8,} rows')

In [ ]:
competitions = pd.read_sql_query('SELECT * FROM competitions ORDER BY competition_name, season_name', conn)
competitions.head(20)

## Top scorers in a competition (default: World Cup 2018)

In [ ]:
COMP_ID, SEASON_ID = 43, 3
sql = '''
WITH all_team_goals AS (
    SELECT home_team_id AS team_id, home_score AS goals FROM matches WHERE competition_id=? AND season_id=?
    UNION ALL
    SELECT away_team_id AS team_id, away_score AS goals FROM matches WHERE competition_id=? AND season_id=?
)
SELECT t.team_name, SUM(g.goals) AS total_goals, COUNT(*) AS matches_played
FROM all_team_goals g JOIN teams t ON t.team_id=g.team_id
GROUP BY t.team_name ORDER BY total_goals DESC LIMIT 10
'''
pd.read_sql_query(sql, conn, params=(COMP_ID, SEASON_ID, COMP_ID, SEASON_ID))

## Elo ratings

In [ ]:
from src.elo import fit_ratings
matches_in_order = pd.read_sql_query(
    'SELECT * FROM matches WHERE competition_id=? AND season_id=? AND home_score IS NOT NULL ORDER BY match_date, kick_off',
    conn, params=(COMP_ID, SEASON_ID))
ratings = fit_ratings(matches_in_order.to_dict('records'))
teams_df = pd.read_sql_query('SELECT * FROM teams', conn)
(pd.DataFrame([{'team_id': k, 'rating': v} for k, v in ratings.items()])
    .merge(teams_df, on='team_id').sort_values('rating', ascending=False).head(15))

## Predict a match (Poisson)

In [ ]:
from src.poisson import compute_team_strengths, expected_goals, match_outcome_probabilities
matches_records = matches_in_order.to_dict('records')
strengths, lh, la = compute_team_strengths(matches_records)
HOME_TEAM_ID, AWAY_TEAM_ID = 771, 785  # France vs Croatia (WC18 final)
lam_h, lam_a = expected_goals(HOME_TEAM_ID, AWAY_TEAM_ID, strengths, lh, la)
result = match_outcome_probabilities(lam_h, lam_a)
print(f'Expected goals: {lam_h:.2f} - {lam_a:.2f}')
print(f'Home win: {result["home_win"]:.1%} | Draw: {result["draw"]:.1%} | Away win: {result["away_win"]:.1%}')

## Season simulation (Monte Carlo)

In [ ]:
from src.monte_carlo import run_simulations
midpoint = len(matches_records) // 2
completed, remaining = matches_records[:midpoint], matches_records[midpoint:]
s_partial, lh, la = compute_team_strengths(completed)
summary = run_simulations(completed, remaining, s_partial, lh, la, n_simulations=5000)
(pd.DataFrame(summary.values()).merge(teams_df, on='team_id')
    .sort_values('avg_position')[['team_name','avg_position','avg_points','p_champion']].head(15))

In [ ]:
conn.close()